In [41]:
import torch
import numpy as np
from torch import nn
import torch.nn.functional as F

In [42]:
device = 'cuda'

In [43]:
class BlockLinear(nn.Module):
    def __init__(self, input_size, output_size, num_blocks=8):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(num_blocks, input_size // num_blocks, output_size // num_blocks))
        self.bias = nn.Parameter(torch.zeros(1, output_size))
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.bias)
    
    def forward(self, x):
        W = torch.block_diag(*self.weight)
        x = x @ W + self.bias
        return x
    
class BlockLinearAlt(nn.Module):
    def __init__(self, input_size, output_size, num_blocks=8):
        super().__init__()
        self.networks = nn.ModuleList([nn.Linear(input_size // num_blocks, output_size // num_blocks) for _ in range(num_blocks)])
        self.num_blocks = num_blocks
    
    def forward(self, x):
        output = []
        x_chunks = torch.split(x, x.shape[-1] // self.num_blocks, dim=-1)
        for i in range(self.num_blocks):
            output.append(self.networks[i](x_chunks[i]))
        return torch.cat(output, -1)

In [46]:
b = 1024
n = 2048
m = 8192
f = BlockLinear(n, m).to(device)
optim = torch.optim.Adam(f.parameters(), 1e-4)
w_init = f.weight.clone()
x = torch.randn((b, n)).to(device)
y = torch.randn((b, m)).to(device)
for _ in range(1000):
    y_ = f(x)
loss = F.mse_loss(y, y_)
loss.backward()
optim.step()
optim.zero_grad()
w_new = f.weight.clone()

In [47]:
f = BlockLinearAlt(n, m).to(device)
optim = torch.optim.Adam(f.parameters(), 1e-4)
x = torch.randn((b, n)).to(device)
y = torch.randn((b, m)).to(device)
for _ in range(1000):
    y_ = f(x)
loss = F.mse_loss(y, y_)
loss.backward()
optim.step()
optim.zero_grad()

In [40]:
y_.shape

torch.Size([1024, 8192])